# Join DistribAI (Kaggle)

Burst GPU worker for a **private orchestrator** — no backend source required.

1. Add secrets in **Kaggle → Notebook → Add-ons → Secrets** (optional): `DISTRIBAI_INVITE_CODE`
2. Set operator endpoints in the config cell.
3. Run all cells.

See `docs/guides/ephemeral-compute-colab-kaggle.md` in the public grid repo.

In [ ]:
# --- Operator join kit ---
ORCHESTRATOR_URL = "your-operator.example.com:50051"
ADMIN_URL = "https://your-operator.example.com"
PUBLIC_GRID_REPO = "https://github.com/naxium-oss/DistribAI.git"
GRPC_USE_TLS = "true"
GRPC_TLS_CA = "/kaggle/working/orchestrator-ca.pem"  # add CA as Kaggle dataset if needed

DISTRIBAI_INVITE_CODE = ""
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    DISTRIBAI_INVITE_CODE = user_secrets.get_secret("DISTRIBAI_INVITE_CODE")
except Exception:
    pass

In [ ]:
import os
import subprocess
import sys

if not os.path.isdir("distribai-public"):
    subprocess.check_call(["git", "clone", "--depth", "1", PUBLIC_GRID_REPO, "distribai-public"])
os.chdir("distribai-public")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-worker.txt"])

In [ ]:
import os

os.environ["ORCHESTRATOR_URL"] = ORCHESTRATOR_URL
os.environ["ADMIN_URL"] = ADMIN_URL
os.environ["DISTRIBAI_INVITE_CODE"] = DISTRIBAI_INVITE_CODE
os.environ["DISTRIBAI_EPHEMERAL"] = "1"
os.environ["GRPC_USE_TLS"] = GRPC_USE_TLS
if os.path.isfile(GRPC_TLS_CA):
    os.environ["GRPC_TLS_CA"] = GRPC_TLS_CA

print("Joining grid as ephemeral worker →", ORCHESTRATOR_URL)

In [ ]:
!python -m worker.src.daemon.run